In [1]:
import sys
import os

sys.path.insert(0, os.path.abspath("/home/ElasticNotebook"))
%load_ext ElasticNotebook
import pickle
import cudf

Enabled rmm statistics


In [2]:
%load_ext cudf.pandas

/opt/conda/envs/rapids-25.02/lib/python3.10/site-packages/cudf/pandas/__init__.py:65: UserWarning: cudf.pandas detected an already configured memory resource, ignoring 'CUDF_PANDAS_RMM_MODE'=managed_pool
  warnings.warn(


In [3]:
%LoadCheckpoint /home/dias-benchmarks/notebooks/erikbruin/nlp-on-student-writing-eda/src/annotated/checkpoints/post_cell_19.pickle

trying: ['i_3', 'i_1']
me:  13
me:  11
trying: ['i_3', 'i_1']
me:  13
me:  11
trying: ['train_first_last']
me:  11
trying: ['plt']
me:  0
trying: ['orig_output']
me:  21
trying: ['style']
me:  0
trying: ['CountVectorizer']
me:  0
trying: ['counter']
me:  11
trying: ['IREWR_plug_2']
me:  16
trying: ['FuncFormatter']
me:  0
trying: ['ax2']
me:  8
trying: ['ax']
me:  11
trying: ['warnings']
me:  0
trying: ['sample_submission']
me:  2
trying: ['ax1']
me:  8
trying: ['np']
me:  0
trying: ['sample_discourse_id']
me:  2
trying: ['cols_to_display']
me:  14
trying: ['all_gaps']
me:  20
trying: ['last_ones']
me:  20
trying: ['train']
me:  14
trying: ['av_per_essay']
me:  20
trying: ['total_gaps']
me:  20
trying: ['IREWR_plug_1']
me:  17
trying: ['IREWR_tmp2']
me:  20
trying: ['IREWR_tmp']
me:  20
trying: ['cols_to_merge']
me:  13
trying: ['glob']
me:  0
trying: ['add_gap_rows']
me:  20
trying: ['pd']
me:  0
trying: ['test_txt']
me:  1
trying: ['myid']
me:  12
trying: ['stopwords']
me:  1
trying:

In [4]:
%%RecordEvent
%%time
### cell 20 ###

def add_gap_rows_2(essay):
    import numpy as np
    import pandas as pd

    cols_to_keep = ['discourse_start', 'discourse_end', 'discourse_type', 'gap_length', 'gap_end_length']
    # pull only the relevant rows and columns
    df_essay = train.query(f"id == '{essay}'")[cols_to_keep].reset_index(drop=True)
    print(df_essay)

    # vectorized computation of gaps between segments
    # shift the end positions down by one to align with the next row
    prev_ends = df_essay['discourse_end'].shift(1)
    # only consider rows 1..N-1 where gap_length>0
    mask_between = (df_essay['gap_length'] > 0) & (df_essay.index > 0)
    if mask_between.any():
        df_between = df_essay.loc[mask_between, []].copy()
        df_between['discourse_start'] = prev_ends[mask_between] + 1
        df_between['discourse_end']   = df_essay.loc[mask_between, 'discourse_start'] - 1
        df_between['discourse_type']  = 'Nothing'
        df_between['gap_length']      = np.nan
        df_between['gap_end_length']  = np.nan
        # append the new gap rows
        df_essay = pd.concat([df_essay, df_between], ignore_index=True)

    # sort all segments by start
    df_essay = df_essay.sort_values('discourse_start').reset_index(drop=True)

    # handle a trailing gap at the end if present
    last_gap_end = df_essay['gap_end_length'].iloc[-1]
    if last_gap_end > 0:
        start = df_essay['discourse_end'].iloc[-1] + 1
        end = start + last_gap_end
        df_tail = pd.DataFrame({
            'discourse_start': [start],
            'discourse_end':   [end],
            'discourse_type':  ['Nothing'],
            'gap_length':      [np.nan],
            'gap_end_length':  [np.nan]
        })
        df_essay = pd.concat([df_essay, df_tail], ignore_index=True)

    return df_essay


CPU times: user 4 µs, sys: 0 ns, total: 4 µs
Wall time: 7.39 µs


In [5]:
%Checkpoint /home/dias-benchmarks/notebooks/erikbruin/nlp-on-student-writing-eda/src/rewritten/cell_19/checkpoints/post_cell_20_try_4.pickle

migration speed (bps): 758778687.2306055
---------------------------
variables to migrate:
last_ones 48
tqdm 1072
sample_submission 48
myword 28
train_first_last 48
len_dict 589920
i_1 28
cols_to_display 120
word_dict 589920
counter 28
glob 144
mylen 28
ax 48
IREWR_plug_2 61
train_first 48
add_gap_rows_2 144
train_last 48
stopwords 48
plt 72
train 48
style 72
CountVectorizer 1072
sample_discourse_id 32
FuncFormatter 1072
warnings 72
pd 72
cols_to_merge 120
ax2 48
i_3 28
ax1 48
np 72
orig_output 48
test_txt 120
av_per_essay 48
add_gap_rows 144
train_txt 126104
IREWR_plug_1 61
IREWR_tmp2 48
myid 61
total_gaps 48
data 2813
IREWR_tmp 48
txt_file 208
all_gaps 48
t 166
---------------------------
variables to recompute:
[]
---------------------------
cells to recompute:
[]
Checkpoint saved to: /home/dias-benchmarks/notebooks/erikbruin/nlp-on-student-writing-eda/src/rewritten/cell_19/checkpoints/post_cell_20_try_4.pickle


In [6]:
%PrintCellInfo opt_cell_exec_info

======= Cell 0 =======
Input variables []
Active variables ['train', 'sample_submission', 'train_txt']
Intermediate variables ['test_txt', 'stopwords']
Future variables []
Modified dataframes
======= Cell 1 =======
Input variables []
Active variables ['train', 'sample_discourse_id']
Intermediate variables ['sample_submission']
Future variables ['train_txt']
Modified dataframes
  train
    Input columns: set()
    Changed columns: {'discourse_end', 'id', 'discourse_id', 'discourse_type_num', 'discourse_text', 'discourse_start', 'discourse_type', 'predictionstring'}
    Created columns: set()
    Deleted columns: set()
  sample_submission
    Input columns: set()
    Changed columns: {'predictionstring', 'id', 'class'}
    Created columns: set()
    Deleted columns: set()
======= Cell 2 =======
Input variables []
Active variables ['train', 'cols_to_display']
Intermediate variables []
Future variables ['train_txt', 'sample_discourse_id']
Modified dataframes
  train
    Input columns: set(

In [7]:
with open(
    "/home/dias-benchmarks/notebooks/erikbruin/nlp-on-student-writing-eda/src/opt_cell_exec_info_20_try_4.pkl",
    "wb",
) as f:
    pickle.dump(opt_cell_exec_info[20], f)

In [8]:
opt_output = Out.get(4)

In [9]:
%LoadCheckpoint /home/dias-benchmarks/notebooks/erikbruin/nlp-on-student-writing-eda/src/annotated/checkpoints/post_cell_20.pickle

import numpy as np
from elastic.core.common.pandas import is_type_styler

is_orig_output_pd = isinstance(orig_output, (pd.Series, pd.DataFrame, pd.Index))
is_opt_output_pd = isinstance(opt_output, (pd.Series, pd.DataFrame, pd.Index))
is_orig_output_array = isinstance(
    orig_output, (cudf.pandas._wrappers.numpy.ndarray, np.ndarray)
)
is_opt_output_array = isinstance(
    opt_output, (cudf.pandas._wrappers.numpy.ndarray, np.ndarray)
)
is_orig_output_styler = is_type_styler(type(orig_output))
is_opt_output_styler = is_type_styler(type(opt_output))
if is_orig_output_styler and is_opt_output_styler:
    assert orig_output.to_html() == opt_output.to_html()
elif is_orig_output_styler:
    assert orig_output.to_html() == opt_output.to_html()
elif is_opt_output_styler:
    assert opt_output.to_html() == orig_output

if is_orig_output_pd and is_opt_output_pd:
    assert orig_output.equals(opt_output)
# TODO(jie): this is a hack.
elif (
    (is_orig_output_pd or is_opt_output_pd)
    and (is_orig_output_array or is_opt_output_array)
) or (is_orig_output_array and is_opt_output_array):
    assert list(orig_output) == list(opt_output)
else:
    assert orig_output == opt_output

trying: ['i_3', 'i_1']
me:  13
me:  11
trying: ['i_3', 'i_1']
me:  13
me:  11
trying: ['tqdm']
me:  0
trying: ['sample_submission']
me:  2
trying: ['myword']
me:  12
trying: ['train_first_last']
me:  11
trying: ['len_dict']
me:  12
trying: ['last_ones']
me:  22
trying: ['cols_to_display']
me:  14
trying: ['word_dict']
me:  12
trying: ['counter']
me:  11
trying: ['glob']
me:  0
trying: ['mylen']
me:  12
trying: ['ax']
me:  11
trying: ['IREWR_plug_2']
me:  16
trying: ['add_gap_rows_2']
me:  22
trying: ['train_first']
me:  11
trying: ['train_last']
me:  11
trying: ['orig_output']
me:  23
trying: ['train']
me:  14
trying: ['stopwords']
me:  1
trying: ['plt']
me:  0
trying: ['style']
me:  0
trying: ['CountVectorizer']
me:  0
trying: ['sample_discourse_id']
me:  2
trying: ['FuncFormatter']
me:  0
trying: ['warnings']
me:  0
trying: ['print_colored_essay']
me:  22
trying: ['pd']
me:  0
trying: ['cols_to_merge']
me:  13
trying: ['ax2']
me:  8
trying: ['ax1']
me:  8
trying: ['np']
me:  0
trying